In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 09_locked_official_test_evaluation.ipynb
# COMPLETE CORRECTED SINGLE-CELL CODE
# ============================================================
#
# Route A locked official-test evaluation.
#
# FIRST official-test access in the revised workflow.
#
# BEFORE defining/accessing the official test directory:
#   1. verify Notebook 08 lock fingerprint
#   2. recompute lock fingerprint
#   3. verify locked artifacts
#   4. verify all 30 checkpoint SHA256 hashes
#
# LOCKED PRIMARY:
#   Swin
#   15 checkpoints = 3 seeds x 5 folds
#
# LOCKED COMPARATOR:
#   ViT
#   15 checkpoints = 3 seeds x 5 folds
#
# Inference:
#   original + horizontal-flip TTA per checkpoint
#   equal mean of 15 checkpoint probabilities
#
# Primary derived patient reference:
#   modal_low
#
# Locked patient aggregation:
#   mean_probability_argmax
#
# Primary endpoint:
#   patient QWK
#
# Secondary:
#   patient MAE
#   patient macro F1
#   patient accuracy
#
# Uncertainty:
#   5,000 stratified patient bootstrap replicates
#
# IMPORTANT:
# After official test evaluation, no tuning/reselection.
# ============================================================


# ============================================================
# 0. Install exact dependencies
# ============================================================

%pip -q install timm==1.0.28 scikit-learn==1.6.1 pandas==2.2.2 Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import io
import os
import re
import json
import time
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torchvision
import timm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    balanced_accuracy_score,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 3. PRE-TEST LOCK PATHS
#
# IMPORTANT:
# The official test path is NOT defined here.
# ============================================================

REVISION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
)

LOCK_ROOT = (
    REVISION_ROOT
    / "08_final_model_lock_before_test"
)

FINAL_LOCK_PATH = (
    LOCK_ROOT
    / "final_model_lock_before_test.json"
)

CHECKPOINT_MANIFEST_PATH = (
    LOCK_ROOT
    / "locked_checkpoint_manifest.csv"
)

LOCK_INTEGRITY_MANIFEST_PATH = (
    LOCK_ROOT
    / "pretest_lock_output_integrity_manifest.csv"
)

READY_FLAG_PATH = (
    LOCK_ROOT
    / "READY_FOR_LOCKED_OFFICIAL_TEST.txt"
)

OUTPUT_ROOT = (
    REVISION_ROOT
    / "09_locked_official_test_evaluation"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_PRETEST_LOCK_FINGERPRINT = (
    "8852c67de6e530e8e6e15b51e94d68be8f3ddffb6d364b328791cf5269e7ef91"
)


for required_path in [
    FINAL_LOCK_PATH,
    CHECKPOINT_MANIFEST_PATH,
    LOCK_INTEGRITY_MANIFEST_PATH,
    READY_FLAG_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Notebook 08 artifact is missing:\n{required_path}"
        )


print("=" * 125)
print("STEP 1 — VERIFY PRE-TEST LOCK BEFORE OFFICIAL-TEST ACCESS")
print("=" * 125)


# ============================================================
# 4. General I/O helpers
# ============================================================

DRIVE_IO_MAX_RETRIES = 6
DRIVE_IO_INITIAL_DELAY_SECONDS = 1.0


def temporary_path(path):
    path = Path(path)
    return path.with_name(
        path.name + ".temporary"
    )


def retry_delay(
    attempt_index,
    initial_delay,
    maximum_delay=10.0,
):
    return min(
        maximum_delay,
        initial_delay * (attempt_index + 1),
    )


def atomic_json_save(
    payload,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            default=str,
        )

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    dataframe.to_csv(
        temp_path,
        index=index,
    )

    os.replace(
        temp_path,
        path,
    )


def atomic_text_save(
    text,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            text
        )

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def file_sha256(
    path,
    block_size=8 * 1024 * 1024,
):
    path = Path(path)

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary,
):
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_torch_load(
    path,
    map_location="cpu",
):
    path = Path(path)

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        try:
            try:
                return torch.load(
                    path,
                    map_location=map_location,
                    weights_only=False,
                )

            except TypeError:
                return torch.load(
                    path,
                    map_location=map_location,
                )

        except OSError as error:
            last_error = error

            delay = retry_delay(
                attempt_index,
                DRIVE_IO_INITIAL_DELAY_SECONDS,
            )

            print(
                "Checkpoint read retry:",
                path,
            )

            time.sleep(
                delay
            )

    raise OSError(
        f"Could not load checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_package_version(
    package_name,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


for temporary_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        temporary_file.unlink()

    except Exception:
        pass


# ============================================================
# 5. Verify Notebook 08 final lock fingerprint
# ============================================================

with open(
    FINAL_LOCK_PATH,
    "r",
    encoding="utf-8",
) as file:
    FINAL_LOCK = json.load(
        file
    )


observed_lock_fingerprint = FINAL_LOCK.get(
    "pretest_lock_fingerprint_sha256"
)


if observed_lock_fingerprint != EXPECTED_PRETEST_LOCK_FINGERPRINT:
    raise RuntimeError(
        "\nPRE-TEST LOCK FINGERPRINT MISMATCH.\n\n"
        f"Expected:\n{EXPECTED_PRETEST_LOCK_FINGERPRINT}\n\n"
        f"Observed:\n{observed_lock_fingerprint}\n\n"
        "OFFICIAL TEST ACCESS ABORTED."
    )


immutable_lock_payload = {
    key: value
    for key, value in FINAL_LOCK.items()
    if key not in {
        "created_utc",
        "pretest_lock_fingerprint_sha256",
    }
}


recomputed_lock_fingerprint = dictionary_sha256(
    immutable_lock_payload
)


if recomputed_lock_fingerprint != EXPECTED_PRETEST_LOCK_FINGERPRINT:
    raise RuntimeError(
        "\nNotebook 08 lock contents no longer reproduce "
        "the locked SHA256 fingerprint.\n\n"
        "OFFICIAL TEST ACCESS ABORTED."
    )


if FINAL_LOCK.get("status") != "PRETEST_LOCKED":
    raise RuntimeError(
        "Notebook 08 status is not PRETEST_LOCKED."
    )


print(
    "Pre-test lock fingerprint verified:"
)

print(
    EXPECTED_PRETEST_LOCK_FINGERPRINT
)


# ============================================================
# 6. Verify Notebook 08 output integrity
# ============================================================

lock_integrity_df = pd.read_csv(
    LOCK_INTEGRITY_MANIFEST_PATH
)


for _, row in lock_integrity_df.iterrows():
    artifact_path = Path(
        row["path"]
    )

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Locked artifact is missing:\n{artifact_path}"
        )

    expected_hash = str(
        row["sha256"]
    )

    observed_hash = file_sha256(
        artifact_path
    )

    if observed_hash != expected_hash:
        raise RuntimeError(
            "\nNotebook 08 artifact integrity failure:\n"
            f"{artifact_path}\n\n"
            f"Expected: {expected_hash}\n"
            f"Observed: {observed_hash}\n\n"
            "OFFICIAL TEST ACCESS ABORTED."
        )


print(
    "Notebook 08 lock-output integrity verified."
)


# ============================================================
# 7. Verify all 30 locked checkpoint hashes
#    BEFORE official-test path definition
# ============================================================

checkpoint_manifest_df = pd.read_csv(
    CHECKPOINT_MANIFEST_PATH
)


if len(checkpoint_manifest_df) != 30:
    raise RuntimeError(
        "Exactly 30 locked checkpoints were expected."
    )


checkpoint_counts = (
    checkpoint_manifest_df[
        "backbone_name"
    ]
    .value_counts()
    .to_dict()
)


if checkpoint_counts.get("Swin") != 15:
    raise RuntimeError(
        "Expected exactly 15 Swin checkpoints."
    )


if checkpoint_counts.get("ViT") != 15:
    raise RuntimeError(
        "Expected exactly 15 ViT checkpoints."
    )


print(
    "\nVerifying all 30 checkpoint SHA256 hashes..."
)


checkpoint_verification_rows = []


ordered_checkpoint_manifest_df = (
    checkpoint_manifest_df
    .sort_values(
        [
            "backbone_name",
            "training_seed",
            "fold",
        ]
    )
    .reset_index(
        drop=True
    )
)


for _, row in ordered_checkpoint_manifest_df.iterrows():
    checkpoint_path = Path(
        row["checkpoint_path"]
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Locked checkpoint missing:\n{checkpoint_path}"
        )

    observed_size = int(
        checkpoint_path.stat().st_size
    )

    expected_size = int(
        row["checkpoint_size_bytes"]
    )

    if observed_size != expected_size:
        raise RuntimeError(
            "\nLocked checkpoint file-size mismatch:\n"
            f"{checkpoint_path}"
        )

    observed_hash = file_sha256(
        checkpoint_path
    )

    expected_hash = str(
        row["checkpoint_sha256"]
    )

    if observed_hash != expected_hash:
        raise RuntimeError(
            "\nLOCKED CHECKPOINT SHA256 MISMATCH:\n"
            f"{checkpoint_path}\n\n"
            f"Expected: {expected_hash}\n"
            f"Observed: {observed_hash}\n\n"
            "OFFICIAL TEST ACCESS ABORTED."
        )

    checkpoint_verification_rows.append({
        "backbone_name": str(
            row["backbone_name"]
        ),

        "training_seed": int(
            row["training_seed"]
        ),

        "fold": int(
            row["fold"]
        ),

        "checkpoint_path": str(
            checkpoint_path
        ),

        "checkpoint_sha256": (
            observed_hash
        ),

        "verified": True,
    })

    print(
        "  verified",
        row["backbone_name"],
        "| seed",
        int(row["training_seed"]),
        "| fold",
        int(row["fold"]),
    )


checkpoint_verification_df = pd.DataFrame(
    checkpoint_verification_rows
)


atomic_csv_save(
    checkpoint_verification_df,

    OUTPUT_ROOT
    / "verified_locked_checkpoint_manifest.csv",

    index=False,
)


print(
    "\nALL 30 CHECKPOINTS VERIFIED."
)

print(
    "Official test path has NOT yet been defined."
)


# ============================================================
# 8. Read locked inference settings
# ============================================================

PRIMARY_SYSTEM = FINAL_LOCK[
    "primary_system"
]

COMPARATOR_SYSTEM = FINAL_LOCK[
    "prespecified_comparator_system"
]

MODEL_RECONSTRUCTION = FINAL_LOCK[
    "model_reconstruction"
]

PREPROCESSING_PLAN = FINAL_LOCK[
    "evaluation_preprocessing"
]

PATIENT_REFERENCE_PLAN = FINAL_LOCK[
    "patient_reference"
]

PATIENT_AGGREGATION_PLAN = FINAL_LOCK[
    "patient_aggregation"
]

BOOTSTRAP_PLAN = FINAL_LOCK[
    "bootstrap_plan"
]


if PRIMARY_SYSTEM["backbone"] != "Swin":
    raise RuntimeError(
        "Locked primary backbone is not Swin."
    )


if COMPARATOR_SYSTEM["backbone"] != "ViT":
    raise RuntimeError(
        "Locked comparator backbone is not ViT."
    )


if (
    PATIENT_REFERENCE_PLAN[
        "primary_reference_name"
    ]
    !=
    "modal_low"
):
    raise RuntimeError(
        "Locked patient reference is not modal_low."
    )


if (
    PATIENT_AGGREGATION_PLAN[
        "primary_rule"
    ]
    !=
    "mean_probability_argmax"
):
    raise RuntimeError(
        "Locked patient aggregation differs from "
        "mean_probability_argmax."
    )


BOOTSTRAP_REPETITIONS = int(
    BOOTSTRAP_PLAN[
        "number_of_replicates"
    ]
)

BOOTSTRAP_SEED = int(
    BOOTSTRAP_PLAN[
        "random_seed"
    ]
)


# ============================================================
# 9. OFFICIAL TEST ACCESS BEGINS HERE
# ============================================================

print("\n" + "=" * 125)

print(
    "STEP 2 — LOCK VERIFIED; OFFICIAL TEST ACCESS NOW BEGINS"
)

print("=" * 125)


EXPECTED_TEST_IMAGE_COUNT = 1686
EXPECTED_TEST_PATIENT_COUNT = 86


LIMUC_DATA_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC')
)


# ------------------------------------------------------------
# Optional manual override
# ------------------------------------------------------------
#
# Leave as None if the official test directory can be found.
#
# If automatic discovery fails, change ONLY this variable to
# the official LIMUC test directory and rerun the IDENTICAL
# locked analysis.
#
MANUAL_OFFICIAL_TEST_ROOT = None


SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
}


def enumerate_images(
    root,
):
    root = Path(root)

    if not root.exists():
        return []

    image_paths = [
        path
        for path in root.rglob("*")
        if (
            path.is_file()
            and
            path.suffix.lower()
            in
            SUPPORTED_IMAGE_EXTENSIONS
        )
    ]

    return sorted(
        image_paths
    )


# ============================================================
# 10. Locate official test directory
# ============================================================

if MANUAL_OFFICIAL_TEST_ROOT is not None:
    OFFICIAL_TEST_ROOT = Path(
        MANUAL_OFFICIAL_TEST_ROOT
    )

    if not OFFICIAL_TEST_ROOT.exists():
        raise FileNotFoundError(
            f"Manual test path does not exist:\n{OFFICIAL_TEST_ROOT}"
        )

else:
    preferred_candidates = [
        LIMUC_DATA_ROOT
        / "test_set",

        LIMUC_DATA_ROOT
        / "test_sets",

        LIMUC_DATA_ROOT
        / "test",

        LIMUC_DATA_ROOT
        / "testing_set",

        LIMUC_DATA_ROOT
        / "official_test_set",
    ]

    exact_matches = []

    for candidate in preferred_candidates:
        if not candidate.exists():
            continue

        images = enumerate_images(
            candidate
        )

        print(
            "Candidate:",
            candidate,
            "| images:",
            len(images),
        )

        if len(images) == EXPECTED_TEST_IMAGE_COUNT:
            exact_matches.append(
                candidate
            )

    if exact_matches:
        OFFICIAL_TEST_ROOT = exact_matches[0]

    else:
        fallback_matches = []

        for candidate in LIMUC_DATA_ROOT.iterdir():
            if not candidate.is_dir():
                continue

            if candidate.name in {
                "ViT2",
                "train_and_validation_sets",
            }:
                continue

            if "test" not in candidate.name.lower():
                continue

            images = enumerate_images(
                candidate
            )

            if len(images) == EXPECTED_TEST_IMAGE_COUNT:
                fallback_matches.append(
                    candidate
                )

        if len(fallback_matches) != 1:
            raise RuntimeError(
                "\nCould not uniquely identify the official "
                "test directory.\n\n"
                f"Matches: {[str(x) for x in fallback_matches]}\n\n"
                "Set MANUAL_OFFICIAL_TEST_ROOT to the known "
                "official test directory and rerun this same cell."
            )

        OFFICIAL_TEST_ROOT = fallback_matches[0]


official_test_image_paths = enumerate_images(
    OFFICIAL_TEST_ROOT
)


if len(
    official_test_image_paths
) != EXPECTED_TEST_IMAGE_COUNT:
    raise RuntimeError(
        "\nOfficial test image-count mismatch.\n"
        f"Expected: {EXPECTED_TEST_IMAGE_COUNT}\n"
        f"Observed: {len(official_test_image_paths)}"
    )


print(
    "\nOfficial test root:"
)

print(
    OFFICIAL_TEST_ROOT
)

print(
    "Official test images:",
    len(official_test_image_paths),
)


# ============================================================
# 11. Locate development manifest from locked provenance
# ============================================================

development_manifest_source_path = None


for provenance_record in FINAL_LOCK[
    "source_artifact_provenance"
]:
    if (
        provenance_record[
            "artifact_name"
        ]
        ==
        "development_manifest"
    ):
        development_manifest_source_path = Path(
            provenance_record[
                "path"
            ]
        )

        break


if development_manifest_source_path is None:
    raise RuntimeError(
        "Development manifest is missing from "
        "Notebook 08 provenance."
    )


if not development_manifest_source_path.exists():
    raise FileNotFoundError(
        f"Development manifest not found:\n"
        f"{development_manifest_source_path}"
    )


development_manifest_df = pd.read_csv(
    development_manifest_source_path
)


for column in [
    "label",
    "patient_id",
]:
    development_manifest_df[
        column
    ] = (
        development_manifest_df[
            column
        ]
        .astype(int)
    )


# ============================================================
# 12. Infer patient-ID and label parsing rules
#     using DEVELOPMENT DATA only
# ============================================================

def numeric_tokens(
    relative_path,
    source,
):
    path = Path(
        str(relative_path)
    )

    if source == "stem":
        text = path.stem

    elif source == "parent":
        text = "/".join(
            path.parts[:-1]
        )

    elif source == "full":
        text = (
            str(relative_path)
            .replace(
                "\\",
                "/",
            )
        )

    else:
        raise ValueError(
            f"Unknown numeric-token source: {source}"
        )

    return re.findall(
        r"\d+",
        text,
    )


def apply_numeric_rule(
    relative_path,
    rule,
):
    tokens = numeric_tokens(
        relative_path,
        rule["source"],
    )

    if not tokens:
        return None

    index = int(
        rule["index"]
    )

    try:
        token = tokens[
            index
        ]

    except IndexError:
        return None

    return int(
        token
    )


def infer_numeric_rule(
    dataframe,
    target_column,
    source_priority,
):
    target_values = (
        dataframe[
            target_column
        ]
        .astype(int)
        .to_numpy()
    )

    candidate_indices = [
        0,
        1,
        2,
        3,
        4,
        5,
        -1,
        -2,
        -3,
        -4,
        -5,
        -6,
    ]

    candidate_rules = []

    for source in source_priority:
        for index in candidate_indices:
            extracted_values = []

            valid = True

            for relative_path in dataframe[
                "relative_path"
            ]:
                extracted = apply_numeric_rule(
                    relative_path,
                    {
                        "source": source,
                        "index": index,
                    },
                )

                if extracted is None:
                    valid = False
                    break

                extracted_values.append(
                    extracted
                )

            if not valid:
                continue

            extracted_values = np.asarray(
                extracted_values,
                dtype=int,
            )

            accuracy = float(
                np.mean(
                    extracted_values
                    ==
                    target_values
                )
            )

            if np.isclose(
                accuracy,
                1.0,
            ):
                candidate_rules.append({
                    "source": source,
                    "index": int(index),
                    "development_exact_accuracy": (
                        accuracy
                    ),
                })

    if not candidate_rules:
        raise RuntimeError(
            f"No exact parsing rule could be inferred for "
            f"'{target_column}' from the development manifest."
        )

    return candidate_rules[0]


PATIENT_ID_PARSING_RULE = infer_numeric_rule(
    development_manifest_df,

    target_column="patient_id",

    source_priority=[
        "stem",
        "parent",
        "full",
    ],
)


LABEL_PARSING_RULE = infer_numeric_rule(
    development_manifest_df,

    target_column="label",

    source_priority=[
        "parent",
        "full",
        "stem",
    ],
)


print(
    "\nPatient-ID parser inferred from development:"
)

print(
    PATIENT_ID_PARSING_RULE
)


print(
    "Label parser inferred from development:"
)

print(
    LABEL_PARSING_RULE
)


# ============================================================
# 13. Build official-test image manifest
# ============================================================

test_manifest_rows = []


for image_path in official_test_image_paths:
    relative_path = image_path.relative_to(
        OFFICIAL_TEST_ROOT
    )

    patient_id = apply_numeric_rule(
        relative_path,
        PATIENT_ID_PARSING_RULE,
    )

    label = apply_numeric_rule(
        relative_path,
        LABEL_PARSING_RULE,
    )

    if patient_id is None:
        raise RuntimeError(
            f"Could not parse patient ID:\n{relative_path}"
        )

    if label is None:
        raise RuntimeError(
            f"Could not parse MES label:\n{relative_path}"
        )

    if label not in {
        0,
        1,
        2,
        3,
    }:
        raise RuntimeError(
            f"Invalid MES label {label} parsed from:\n"
            f"{relative_path}"
        )

    test_manifest_rows.append({
        "path": str(
            image_path
        ),

        "relative_path": str(
            relative_path
        ),

        "filename": (
            image_path.name
        ),

        "patient_id": int(
            patient_id
        ),

        "label": int(
            label
        ),

        "file_size_bytes": int(
            image_path.stat().st_size
        ),
    })


test_manifest_df = (
    pd.DataFrame(
        test_manifest_rows
    )
    .sort_values(
        "path"
    )
    .reset_index(
        drop=True
    )
)


if test_manifest_df[
    "path"
].duplicated().any():
    raise RuntimeError(
        "Duplicate official-test image paths detected."
    )


observed_test_patient_count = int(
    test_manifest_df[
        "patient_id"
    ].nunique()
)


if observed_test_patient_count != EXPECTED_TEST_PATIENT_COUNT:
    raise RuntimeError(
        "\nOfficial test patient-count mismatch.\n"
        f"Expected: {EXPECTED_TEST_PATIENT_COUNT}\n"
        f"Observed: {observed_test_patient_count}\n\n"
        "Do NOT continue until the patient-ID parsing "
        "rule/dataset structure is resolved."
    )


TEST_MANIFEST_PATH = (
    OUTPUT_ROOT
    / "official_test_image_manifest.csv"
)


atomic_csv_save(
    test_manifest_df,
    TEST_MANIFEST_PATH,
    index=False,
)


TEST_MANIFEST_SHA256 = file_sha256(
    TEST_MANIFEST_PATH
)


print(
    "\nOfficial test manifest validated."
)

print(
    "Images:",
    len(test_manifest_df),
)

print(
    "Patients:",
    observed_test_patient_count,
)

print(
    "Image-level class counts:"
)

print(
    test_manifest_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


# ============================================================
# 14. Construct derived patient-reference definitions
# ============================================================

NUM_CLASSES = 4


def modal_class(
    values,
    tie_rule,
):
    counts = np.bincount(
        np.asarray(
            values,
            dtype=int,
        ),

        minlength=NUM_CLASSES,
    )

    maximum_count = counts.max()

    tied_classes = np.flatnonzero(
        counts
        ==
        maximum_count
    )

    if tie_rule == "low":
        return int(
            tied_classes.min()
        )

    if tie_rule == "high":
        return int(
            tied_classes.max()
        )

    raise ValueError(
        f"Unknown modal tie rule: {tie_rule}"
    )


def shannon_entropy_bits(
    values,
):
    counts = np.bincount(
        np.asarray(
            values,
            dtype=int,
        ),

        minlength=NUM_CLASSES,
    )

    probabilities = (
        counts[
            counts > 0
        ]
        /
        counts.sum()
    )

    return float(
        -np.sum(
            probabilities
            *
            np.log2(
                probabilities
            )
        )
    )


patient_reference_rows = []


for patient_id, patient_images in test_manifest_df.groupby(
    "patient_id",
    sort=True,
):
    labels = (
        patient_images[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    counts = np.bincount(
        labels,
        minlength=NUM_CLASSES,
    )

    maximum_count = int(
        counts.max()
    )

    tied_classes = np.flatnonzero(
        counts
        ==
        maximum_count
    )

    median_value = float(
        np.median(
            labels
        )
    )

    patient_reference_rows.append({
        "patient_id": int(
            patient_id
        ),

        "n_images": int(
            len(labels)
        ),

        "n_unique_labels": int(
            len(
                np.unique(
                    labels
                )
            )
        ),

        "minimum_label": int(
            labels.min()
        ),

        "maximum_label": int(
            labels.max()
        ),

        "label_range": int(
            labels.max()
            -
            labels.min()
        ),

        "modal_tie": bool(
            len(tied_classes)
            >
            1
        ),

        "majority_fraction": float(
            maximum_count
            /
            len(labels)
        ),

        "label_entropy": (
            shannon_entropy_bits(
                labels
            )
        ),

        "modal_low": (
            modal_class(
                labels,
                "low",
            )
        ),

        "modal_high": (
            modal_class(
                labels,
                "high",
            )
        ),

        "median_low": int(
            np.floor(
                median_value
            )
        ),

        "median_high": int(
            np.ceil(
                median_value
            )
        ),
    })


patient_reference_df = (
    pd.DataFrame(
        patient_reference_rows
    )
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    patient_reference_df
) != EXPECTED_TEST_PATIENT_COUNT:
    raise RuntimeError(
        "Derived patient-reference table does not "
        "contain exactly 86 patients."
    )


PATIENT_REFERENCE_PATH = (
    OUTPUT_ROOT
    / "official_test_patient_reference_audit.csv"
)


atomic_csv_save(
    patient_reference_df,
    PATIENT_REFERENCE_PATH,
    index=False,
)


primary_reference_counts = (
    patient_reference_df[
        "modal_low"
    ]
    .value_counts()
    .reindex(
        range(
            NUM_CLASSES
        ),
        fill_value=0,
    )
    .sort_index()
)


print(
    "\nPrimary modal_low patient counts:"
)

print(
    primary_reference_counts.to_dict()
)


# ============================================================
# 15. Reference disagreement audit
# ============================================================

REFERENCE_RULES = [
    "modal_low",
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]


reference_disagreement_rows = []


for alternative_reference in [
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]:
    disagreement = (
        patient_reference_df[
            "modal_low"
        ]
        !=
        patient_reference_df[
            alternative_reference
        ]
    )

    reference_disagreement_rows.append({
        "primary_reference": (
            "modal_low"
        ),

        "alternative_reference": (
            alternative_reference
        ),

        "n_disagree": int(
            disagreement.sum()
        ),

        "percent_disagree": float(
            100.0
            *
            disagreement.mean()
        ),
    })


reference_disagreement_df = pd.DataFrame(
    reference_disagreement_rows
)


atomic_csv_save(
    reference_disagreement_df,

    OUTPUT_ROOT
    / "official_test_reference_disagreement.csv",

    index=False,
)


# ============================================================
# 16. Test-set heterogeneity audit
# ============================================================

test_heterogeneity_audit = {
    "n_images": int(
        len(
            test_manifest_df
        )
    ),

    "n_patients": int(
        len(
            patient_reference_df
        )
    ),

    "patients_with_multiple_image_labels": int(
        (
            patient_reference_df[
                "n_unique_labels"
            ]
            >
            1
        ).sum()
    ),

    "patients_with_modal_ties": int(
        patient_reference_df[
            "modal_tie"
        ].sum()
    ),

    "patients_label_range_ge_2": int(
        (
            patient_reference_df[
                "label_range"
            ]
            >=
            2
        ).sum()
    ),

    "patients_label_range_ge_3": int(
        (
            patient_reference_df[
                "label_range"
            ]
            >=
            3
        ).sum()
    ),

    "patients_majority_fraction_lt_0_50": int(
        (
            patient_reference_df[
                "majority_fraction"
            ]
            <
            0.50
        ).sum()
    ),

    "patients_majority_fraction_lt_0_67": int(
        (
            patient_reference_df[
                "majority_fraction"
            ]
            <
            0.67
        ).sum()
    ),

    "patients_entropy_gt_1_bit": int(
        (
            patient_reference_df[
                "label_entropy"
            ]
            >
            1.0
        ).sum()
    ),

    "modal_low_class_counts": {
        str(key): int(value)
        for key, value in (
            primary_reference_counts.items()
        )
    },
}


atomic_json_save(
    test_heterogeneity_audit,

    OUTPUT_ROOT
    / "official_test_dataset_audit.json",
)


# ============================================================
# 17. Local official-test image cache
# ============================================================

LOCAL_TEST_CACHE_ROOT = Path(
    "/content/"
    +
    "LIMUC_official_test_cache_"
    +
    EXPECTED_PRETEST_LOCK_FINGERPRINT[:12]
)


LOCAL_TEST_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


IMAGE_IO_MAX_RETRIES = 12
IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0
IMAGE_IO_MAX_DELAY_SECONDS = 8.0


def local_test_cache_path(
    relative_path,
):
    return (
        LOCAL_TEST_CACHE_ROOT
        /
        Path(
            str(relative_path)
        )
    )


def read_image_bytes_with_retry(
    source_path,
):
    source_path = Path(
        source_path
    )

    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            with open(
                source_path,
                "rb",
            ) as file:
                data = file.read()

            if len(data) == 0:
                raise OSError(
                    "Zero-byte image read."
                )

            return data

        except (
            OSError,
            IOError,
        ) as error:
            last_error = error

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "Transient image-read failure:",
                source_path,
            )

            print(
                f"Retrying in {delay:.1f}s"
            )

            time.sleep(
                delay
            )

    raise OSError(
        f"Could not read image:\n{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


def ensure_test_image_cached(
    source_path,
    relative_path,
):
    local_path = local_test_cache_path(
        relative_path
    )

    if (
        local_path.exists()
        and
        local_path.stat().st_size > 0
    ):
        return local_path

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_bytes = read_image_bytes_with_retry(
        source_path
    )

    temp_path = temporary_path(
        local_path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "wb",
    ) as file:
        file.write(
            image_bytes
        )

    os.replace(
        temp_path,
        local_path,
    )

    return local_path


def open_cached_rgb_image(
    source_path,
    relative_path,
):
    last_error = None

    for _ in range(
        3
    ):
        local_path = ensure_test_image_cached(
            source_path,
            relative_path,
        )

        try:
            with open(
                local_path,
                "rb",
            ) as file:
                image_bytes = file.read()

            with Image.open(
                io.BytesIO(
                    image_bytes
                )
            ) as image:
                image.load()

                return image.convert(
                    "RGB"
                )

        except Exception as error:
            last_error = error

            try:
                local_path.unlink()

            except Exception:
                pass

    raise OSError(
        f"Could not decode official-test image:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


# ============================================================
# 18. Cache official-test images
#     *** SYNTAX ERROR FIXED HERE ***
# ============================================================

print(
    "\nCaching official-test images locally..."
)


for image_index, row in test_manifest_df.iterrows():
    local_path = ensure_test_image_cached(
        row["path"],
        row["relative_path"],
    )

    # Periodically decode an image to verify cache integrity.
    if image_index % 100 == 0:
        with Image.open(
            local_path
        ) as image:
            image.verify()

    if (image_index + 1) % 250 == 0:
        print(
            "Cached",
            image_index + 1,
            "/",
            len(test_manifest_df),
        )


print(
    "Official-test cache ready:"
)

print(
    LOCAL_TEST_CACHE_ROOT
)


# ============================================================
# 19. Locked evaluation preprocessing
# ============================================================

IMAGE_SIZE = int(
    PREPROCESSING_PLAN[
        "image_size"
    ][0]
)


EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE,
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(
            PREPROCESSING_PLAN[
                "normalization_mean"
            ]
        ),

        std=(
            PREPROCESSING_PLAN[
                "normalization_std"
            ]
        ),
    ),
])


# ============================================================
# 20. Official test Dataset
# ============================================================

class OfficialTestDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
    ):
        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

    def __len__(
        self,
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[
            index
        ]

        image = open_cached_rgb_image(
            row["path"],
            row["relative_path"],
        )

        image = EVAL_TRANSFORM(
            image
        )

        return {
            "image": image,

            "path": str(
                row["path"]
            ),

            "patient_id": torch.tensor(
                int(
                    row["patient_id"]
                ),
                dtype=torch.long,
            ),

            "label": torch.tensor(
                int(
                    row["label"]
                ),
                dtype=torch.long,
            ),
        }


TEST_BATCH_SIZE = 8
RUNTIME_NUM_WORKERS = 0


test_dataset = OfficialTestDataset(
    test_manifest_df
)


test_loader = DataLoader(
    test_dataset,

    batch_size=(
        TEST_BATCH_SIZE
    ),

    shuffle=False,

    num_workers=(
        RUNTIME_NUM_WORKERS
    ),

    pin_memory=(
        torch.cuda.is_available()
    ),

    drop_last=False,

    persistent_workers=False,
)


# ============================================================
# 21. Device / AMP
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


USE_MIXED_PRECISION = bool(
    MODEL_RECONSTRUCTION[
        "mixed_precision_inference"
    ]
)


print(
    "\nInference device:",
    device,
)


if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        ),
    )


def autocast_context():
    enabled = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )

    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 22. Prediction validation
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def validate_checkpoint_prediction_df(
    prediction_df,
):
    required_columns = {
        "path",
        "patient_id",
        "y_true",
        *PROBABILITY_COLUMNS,
    }

    if not required_columns.issubset(
        prediction_df.columns
    ):
        return False

    prediction_df = (
        prediction_df
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )

    expected_df = (
        test_manifest_df
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )

    if len(prediction_df) != len(
        expected_df
    ):
        return False

    if not np.array_equal(
        prediction_df[
            "path"
        ].astype(
            str
        ).to_numpy(),

        expected_df[
            "path"
        ].astype(
            str
        ).to_numpy(),
    ):
        return False

    if not np.array_equal(
        prediction_df[
            "patient_id"
        ].astype(
            int
        ).to_numpy(),

        expected_df[
            "patient_id"
        ].astype(
            int
        ).to_numpy(),
    ):
        return False

    if not np.array_equal(
        prediction_df[
            "y_true"
        ].astype(
            int
        ).to_numpy(),

        expected_df[
            "label"
        ].astype(
            int
        ).to_numpy(),
    ):
        return False

    probabilities = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    if not np.isfinite(
        probabilities
    ).all():
        return False

    if not np.allclose(
        probabilities.sum(
            axis=1
        ),
        1.0,
        atol=2e-3,
        rtol=0.0,
    ):
        return False

    return True


# ============================================================
# 23. Resume-safe inference directories
# ============================================================

CHECKPOINT_PREDICTION_ROOT = (
    OUTPUT_ROOT
    / "checkpoint_predictions"
)


CHECKPOINT_PREDICTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 24. Run one locked checkpoint
# ============================================================

@torch.no_grad()
def run_locked_checkpoint(
    checkpoint_record,
):
    backbone_name = str(
        checkpoint_record[
            "backbone_name"
        ]
    )

    training_seed = int(
        checkpoint_record[
            "training_seed"
        ]
    )

    fold = int(
        checkpoint_record[
            "fold"
        ]
    )

    model_name = str(
        checkpoint_record[
            "model_name"
        ]
    )

    checkpoint_path = Path(
        checkpoint_record[
            "checkpoint_path"
        ]
    )

    checkpoint_hash = str(
        checkpoint_record[
            "checkpoint_sha256"
        ]
    )

    output_directory = (
        CHECKPOINT_PREDICTION_ROOT
        /
        backbone_name
        /
        f"seed_{training_seed}"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    prediction_path = (
        output_directory
        /
        f"fold_{fold}_official_test_probabilities.csv"
    )

    metadata_path = (
        output_directory
        /
        f"fold_{fold}_official_test_metadata.json"
    )

    expected_metadata = {
        "pretest_lock_fingerprint": (
            EXPECTED_PRETEST_LOCK_FINGERPRINT
        ),

        "test_manifest_sha256": (
            TEST_MANIFEST_SHA256
        ),

        "backbone_name": (
            backbone_name
        ),

        "model_name": (
            model_name
        ),

        "training_seed": (
            training_seed
        ),

        "fold": (
            fold
        ),

        "checkpoint_sha256": (
            checkpoint_hash
        ),

        "tta": (
            "original_plus_horizontal_flip_probability_mean"
        ),

        "checkpoint_ensemble_weight": (
            1.0
            /
            15.0
        ),
    }

    # --------------------------------------------------------
    # Resume completed inference
    # --------------------------------------------------------

    if (
        prediction_path.exists()
        and
        metadata_path.exists()
    ):
        try:
            with open(
                metadata_path,
                "r",
                encoding="utf-8",
            ) as file:
                existing_metadata = json.load(
                    file
                )

            existing_prediction_df = pd.read_csv(
                prediction_path
            )

            if (
                existing_metadata
                ==
                expected_metadata
                and
                validate_checkpoint_prediction_df(
                    existing_prediction_df
                )
            ):
                print(
                    "Loaded completed inference:",
                    backbone_name,
                    "| seed",
                    training_seed,
                    "| fold",
                    fold,
                )

                return (
                    existing_prediction_df
                    .sort_values(
                        "path"
                    )
                    .reset_index(
                        drop=True
                    )
                )

        except Exception:
            pass

    # --------------------------------------------------------
    # Fresh inference
    # --------------------------------------------------------

    print(
        "\nRunning:",
        backbone_name,
        "| seed",
        training_seed,
        "| fold",
        fold,
    )

    reconstruction_key = (
        backbone_name.lower()
    )

    reconstruction_config = (
        MODEL_RECONSTRUCTION[
            reconstruction_key
        ]
    )

    model = timm.create_model(
        model_name,

        pretrained=False,

        num_classes=int(
            reconstruction_config[
                "num_classes"
            ]
        ),

        drop_rate=float(
            reconstruction_config[
                "drop_rate"
            ]
        ),

        drop_path_rate=float(
            reconstruction_config[
                "drop_path_rate"
            ]
        ),
    )

    checkpoint = safe_torch_load(
        checkpoint_path,
        map_location="cpu",
    )

    if (
        "model_state_dict"
        not in
        checkpoint
    ):
        raise RuntimeError(
            f"model_state_dict missing:\n{checkpoint_path}"
        )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    model = model.to(
        device
    )

    model.eval()

    path_rows = []
    patient_rows = []
    label_rows = []
    probability_rows = []

    for batch in test_loader:
        images = (
            batch[
                "image"
            ]
            .to(
                device,
                non_blocking=True,
            )
        )

        with autocast_context():
            logits_original = model(
                images
            )

            probabilities_original = torch.softmax(
                logits_original,
                dim=1,
            )

            flipped_images = torch.flip(
                images,
                dims=[
                    3
                ],
            )

            logits_flipped = model(
                flipped_images
            )

            probabilities_flipped = torch.softmax(
                logits_flipped,
                dim=1,
            )

            probabilities = (
                probabilities_original
                +
                probabilities_flipped
            ) / 2.0

        path_rows.extend(
            batch[
                "path"
            ]
        )

        patient_rows.extend(
            batch[
                "patient_id"
            ]
            .cpu()
            .numpy()
            .tolist()
        )

        label_rows.extend(
            batch[
                "label"
            ]
            .cpu()
            .numpy()
            .tolist()
        )

        probability_rows.extend(
            probabilities
            .float()
            .cpu()
            .numpy()
            .tolist()
        )

    prediction_df = pd.DataFrame({
        "path": (
            path_rows
        ),

        "patient_id": (
            patient_rows
        ),

        "y_true": (
            label_rows
        ),
    })

    probability_df = pd.DataFrame(
        probability_rows,
        columns=(
            PROBABILITY_COLUMNS
        ),
    )

    prediction_df = pd.concat(
        [
            prediction_df,
            probability_df,
        ],
        axis=1,
    )

    prediction_df[
        "y_pred"
    ] = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
        .argmax(
            axis=1
        )
        .astype(int)
    )

    prediction_df[
        "training_seed"
    ] = training_seed

    prediction_df[
        "fold"
    ] = fold

    prediction_df[
        "backbone_name"
    ] = backbone_name

    prediction_df = (
        prediction_df
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )

    if not validate_checkpoint_prediction_df(
        prediction_df
    ):
        raise RuntimeError(
            "Completed checkpoint prediction "
            "failed validation."
        )

    atomic_csv_save(
        prediction_df,
        prediction_path,
        index=False,
    )

    atomic_json_save(
        expected_metadata,
        metadata_path,
    )

    del model
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return prediction_df


# ============================================================
# 25. Run all 30 locked checkpoints
# ============================================================

checkpoint_prediction_lookup = {
    "Swin": [],
    "ViT": [],
}


for _, checkpoint_record in ordered_checkpoint_manifest_df.iterrows():
    prediction_df = run_locked_checkpoint(
        checkpoint_record
    )

    checkpoint_prediction_lookup[
        str(
            checkpoint_record[
                "backbone_name"
            ]
        )
    ].append(
        prediction_df
    )


if len(
    checkpoint_prediction_lookup[
        "Swin"
    ]
) != 15:
    raise RuntimeError(
        "Expected 15 Swin predictions."
    )


if len(
    checkpoint_prediction_lookup[
        "ViT"
    ]
) != 15:
    raise RuntimeError(
        "Expected 15 ViT predictions."
    )


print(
    "\nAll 30 locked checkpoint inferences completed."
)


# ============================================================
# 26. Build locked 15-checkpoint image ensembles
# ============================================================

def build_locked_ensemble(
    backbone_name,
):
    prediction_frames = (
        checkpoint_prediction_lookup[
            backbone_name
        ]
    )

    base_df = (
        prediction_frames[
            0
        ][
            [
                "path",
                "patient_id",
                "y_true",
            ]
        ]
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    probability_arrays = []

    for frame in prediction_frames:
        frame = (
            frame
            .sort_values(
                "path"
            )
            .reset_index(
                drop=True
            )
        )

        if not np.array_equal(
            frame[
                "path"
            ].astype(
                str
            ).to_numpy(),

            base_df[
                "path"
            ].astype(
                str
            ).to_numpy(),
        ):
            raise RuntimeError(
                "Checkpoint image alignment failure."
            )

        if not np.array_equal(
            frame[
                "patient_id"
            ].astype(
                int
            ).to_numpy(),

            base_df[
                "patient_id"
            ].astype(
                int
            ).to_numpy(),
        ):
            raise RuntimeError(
                "Checkpoint patient alignment failure."
            )

        if not np.array_equal(
            frame[
                "y_true"
            ].astype(
                int
            ).to_numpy(),

            base_df[
                "y_true"
            ].astype(
                int
            ).to_numpy(),
        ):
            raise RuntimeError(
                "Checkpoint label alignment failure."
            )

        probability_arrays.append(
            frame[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )

    stacked_probabilities = np.stack(
        probability_arrays,
        axis=0,
    )

    mean_probabilities = np.mean(
        stacked_probabilities,
        axis=0,
    )

    ensemble_df = base_df.copy()

    for class_index, column_name in enumerate(
        PROBABILITY_COLUMNS
    ):
        ensemble_df[
            column_name
        ] = (
            mean_probabilities[
                :,
                class_index,
            ]
        )

    ensemble_df[
        "y_pred"
    ] = (
        mean_probabilities
        .argmax(
            axis=1
        )
        .astype(int)
    )

    ensemble_df[
        "backbone_name"
    ] = backbone_name

    ensemble_df[
        "n_checkpoints"
    ] = 15

    ensemble_df[
        "ensemble_type"
    ] = (
        "equal_probability_mean_15_checkpoints"
    )

    modal_low_lookup = (
        patient_reference_df
        .set_index(
            "patient_id"
        )[
            "modal_low"
        ]
        .to_dict()
    )

    ensemble_df[
        "patient_y_true"
    ] = (
        ensemble_df[
            "patient_id"
        ]
        .map(
            modal_low_lookup
        )
        .astype(int)
    )

    return (
        ensemble_df
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )


swin_image_df = build_locked_ensemble(
    "Swin"
)


vit_image_df = build_locked_ensemble(
    "ViT"
)


SWIN_IMAGE_PREDICTION_PATH = (
    OUTPUT_ROOT
    / "Swin_official_test_image_predictions_15_checkpoint_ensemble.csv"
)


VIT_IMAGE_PREDICTION_PATH = (
    OUTPUT_ROOT
    / "ViT_official_test_image_predictions_15_checkpoint_ensemble.csv"
)


atomic_csv_save(
    swin_image_df,
    SWIN_IMAGE_PREDICTION_PATH,
    index=False,
)


atomic_csv_save(
    vit_image_df,
    VIT_IMAGE_PREDICTION_PATH,
    index=False,
)


# ============================================================
# 27. Core metric helpers
# ============================================================

def compute_multiclass_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],

        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,

                labels=[
                    0,
                    1,
                    2,
                    3,
                ],

                average="macro",

                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def raw_confusion_df(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],
    )

    return pd.DataFrame(
        matrix,

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def normalized_confusion_df(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],
    ).astype(float)

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    normalized = np.divide(
        matrix,
        row_sums,

        out=np.zeros_like(
            matrix
        ),

        where=(
            row_sums
            !=
            0
        ),
    )

    return pd.DataFrame(
        normalized,

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def per_class_metrics_df(
    y_true,
    y_pred,
    model_name,
    evaluation_level,
):
    precision, recall, f1_values, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,

            labels=[
                0,
                1,
                2,
                3,
            ],

            zero_division=0,
        )
    )

    rows = []

    for class_index in range(
        NUM_CLASSES
    ):
        rows.append({
            "model": (
                model_name
            ),

            "evaluation_level": (
                evaluation_level
            ),

            "mayo_class": (
                int(
                    class_index
                )
            ),

            "precision": float(
                precision[
                    class_index
                ]
            ),

            "recall": float(
                recall[
                    class_index
                ]
            ),

            "f1": float(
                f1_values[
                    class_index
                ]
            ),

            "support": int(
                support[
                    class_index
                ]
            ),
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 28. Locked primary patient aggregation
# ============================================================

def aggregate_mean_probability_argmax(
    image_df,
):
    probability_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )[
            PROBABILITY_COLUMNS
        ]
        .mean()
    )

    metadata_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        metadata_df
        .merge(
            probability_df,

            on="patient_id",

            how="inner",

            validate="one_to_one",
        )
    )

    patient_df[
        "y_pred"
    ] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
        .argmax(
            axis=1
        )
        .astype(int)
    )

    return (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )


swin_patient_df = aggregate_mean_probability_argmax(
    swin_image_df
)


vit_patient_df = aggregate_mean_probability_argmax(
    vit_image_df
)


if len(
    swin_patient_df
) != EXPECTED_TEST_PATIENT_COUNT:
    raise RuntimeError(
        "Swin patient count is not 86."
    )


if len(
    vit_patient_df
) != EXPECTED_TEST_PATIENT_COUNT:
    raise RuntimeError(
        "ViT patient count is not 86."
    )


SWIN_PATIENT_PREDICTION_PATH = (
    OUTPUT_ROOT
    / "Swin_official_test_patient_predictions_locked.csv"
)


VIT_PATIENT_PREDICTION_PATH = (
    OUTPUT_ROOT
    / "ViT_official_test_patient_predictions_locked.csv"
)


atomic_csv_save(
    swin_patient_df,
    SWIN_PATIENT_PREDICTION_PATH,
    index=False,
)


atomic_csv_save(
    vit_patient_df,
    VIT_PATIENT_PREDICTION_PATH,
    index=False,
)


# ============================================================
# 29. Primary point estimates
# ============================================================

model_image_frames = {
    "Swin": swin_image_df,
    "ViT": vit_image_df,
}


model_patient_frames = {
    "Swin": swin_patient_df,
    "ViT": vit_patient_df,
}


main_point_rows = []


for model_name in [
    "Swin",
    "ViT",
]:
    image_df = model_image_frames[
        model_name
    ]

    patient_df = model_patient_frames[
        model_name
    ]

    image_metrics = compute_multiclass_metrics(
        image_df[
            "y_true"
        ],
        image_df[
            "y_pred"
        ],
    )

    patient_metrics = compute_multiclass_metrics(
        patient_df[
            "y_true"
        ],
        patient_df[
            "y_pred"
        ],
    )

    main_point_rows.append({
        "model": (
            model_name
        ),

        "role": (
            "primary"
            if model_name == "Swin"
            else
            "prespecified_comparator"
        ),

        "patient_qwk": (
            patient_metrics[
                "qwk"
            ]
        ),

        "patient_mae": (
            patient_metrics[
                "mae"
            ]
        ),

        "patient_macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),

        "patient_accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),

        "image_qwk": (
            image_metrics[
                "qwk"
            ]
        ),

        "image_mae": (
            image_metrics[
                "mae"
            ]
        ),

        "image_macro_f1": (
            image_metrics[
                "macro_f1"
            ]
        ),

        "image_accuracy": (
            image_metrics[
                "accuracy"
            ]
        ),
    })


main_point_df = pd.DataFrame(
    main_point_rows
)


# ============================================================
# 30. Severe Mayo 0/1 vs 2/3 helper
# ============================================================

def severe_binary_metrics(
    y_true,
    y_pred,
):
    y_true_binary = (
        np.asarray(
            y_true,
            dtype=int,
        )
        >=
        2
    ).astype(int)

    y_pred_binary = (
        np.asarray(
            y_pred,
            dtype=int,
        )
        >=
        2
    ).astype(int)

    matrix = confusion_matrix(
        y_true_binary,
        y_pred_binary,

        labels=[
            0,
            1,
        ],
    )

    tn, fp, fn, tp = (
        matrix.ravel()
    )

    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
        if
        (
            tp
            +
            fn
        )
        >
        0
        else
        np.nan
    )

    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
        if
        (
            tn
            +
            fp
        )
        >
        0
        else
        np.nan
    )

    return {
        "severe_sensitivity": float(
            sensitivity
        ),

        "severe_specificity": float(
            specificity
        ),

        "severe_binary_f1": float(
            f1_score(
                y_true_binary,
                y_pred_binary,
                zero_division=0,
            )
        ),

        "severe_balanced_accuracy": float(
            balanced_accuracy_score(
                y_true_binary,
                y_pred_binary,
            )
        ),

        "severe_binary_accuracy": float(
            accuracy_score(
                y_true_binary,
                y_pred_binary,
            )
        ),
    }


# ============================================================
# 31. Stratified patient bootstrap
# ============================================================

def bootstrap_one_model(
    patient_df,
    model_name,
    repetitions,
    seed,
):
    patient_df = (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    y_true = (
        patient_df[
            "y_true"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_pred = (
        patient_df[
            "y_pred"
        ]
        .to_numpy(
            dtype=int
        )
    )

    class_indices = {
        class_index: np.flatnonzero(
            y_true
            ==
            class_index
        )

        for class_index in range(
            NUM_CLASSES
        )
    }

    for class_index in range(
        NUM_CLASSES
    ):
        if len(
            class_indices[
                class_index
            ]
        ) == 0:
            raise RuntimeError(
                "\nPrimary official-test modal_low class "
                f"{class_index} has zero patients.\n"
                "The locked class-stratified bootstrap "
                "cannot be performed exactly as specified."
            )

    rng = np.random.default_rng(
        seed
    )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_indices = np.concatenate([
            rng.choice(
                class_indices[
                    class_index
                ],

                size=len(
                    class_indices[
                        class_index
                    ]
                ),

                replace=True,
            )

            for class_index in range(
                NUM_CLASSES
            )
        ])

        sampled_true = y_true[
            sampled_indices
        ]

        sampled_pred = y_pred[
            sampled_indices
        ]

        metrics = compute_multiclass_metrics(
            sampled_true,
            sampled_pred,
        )

        precision, recall, class_f1, _ = (
            precision_recall_fscore_support(
                sampled_true,
                sampled_pred,

                labels=[
                    0,
                    1,
                    2,
                    3,
                ],

                zero_division=0,
            )
        )

        severe_metrics = severe_binary_metrics(
            sampled_true,
            sampled_pred,
        )

        row = {
            "bootstrap_index": (
                int(
                    bootstrap_index
                )
            ),

            **metrics,
            **severe_metrics,
        }

        for class_index in range(
            NUM_CLASSES
        ):
            row[
                f"class_{class_index}_precision"
            ] = float(
                precision[
                    class_index
                ]
            )

            row[
                f"class_{class_index}_recall"
            ] = float(
                recall[
                    class_index
                ]
            )

            row[
                f"class_{class_index}_f1"
            ] = float(
                class_f1[
                    class_index
                ]
            )

        bootstrap_rows.append(
            row
        )

        if (
            (
                bootstrap_index
                +
                1
            )
            %
            1000
            ==
            0
        ):
            print(
                model_name,
                "bootstrap:",
                bootstrap_index + 1,
                "/",
                repetitions,
            )

    return pd.DataFrame(
        bootstrap_rows
    )


model_bootstrap_lookup = {}


for model_index, model_name in enumerate(
    [
        "Swin",
        "ViT",
    ]
):
    bootstrap_df = bootstrap_one_model(
        model_patient_frames[
            model_name
        ],

        model_name=(
            model_name
        ),

        repetitions=(
            BOOTSTRAP_REPETITIONS
        ),

        seed=(
            BOOTSTRAP_SEED
            +
            model_index
            *
            10000
        ),
    )

    model_bootstrap_lookup[
        model_name
    ] = bootstrap_df

    atomic_csv_save(
        bootstrap_df,

        OUTPUT_ROOT
        /
        f"{model_name}_official_test_patient_bootstrap_samples.csv",

        index=False,
    )


# ============================================================
# 32. Main marginal bootstrap CIs
# ============================================================

MAIN_PATIENT_METRICS = [
    "qwk",
    "mae",
    "macro_f1",
    "accuracy",
]


marginal_ci_rows = []


for model_name in [
    "Swin",
    "ViT",
]:
    patient_df = model_patient_frames[
        model_name
    ]

    point_metrics = compute_multiclass_metrics(
        patient_df[
            "y_true"
        ],
        patient_df[
            "y_pred"
        ],
    )

    bootstrap_df = model_bootstrap_lookup[
        model_name
    ]

    for metric_name in MAIN_PATIENT_METRICS:
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        marginal_ci_rows.append({
            "model": (
                model_name
            ),

            "metric": (
                metric_name
            ),

            "point_estimate": float(
                point_metrics[
                    metric_name
                ]
            ),

            "ci_low": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "ci_high": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "n_bootstrap": (
                int(
                    len(values)
                )
            ),
        })


marginal_ci_df = pd.DataFrame(
    marginal_ci_rows
)


atomic_csv_save(
    marginal_ci_df,

    OUTPUT_ROOT
    / "official_test_patient_metric_bootstrap_ci.csv",

    index=False,
)


# ============================================================
# 33. Paired Swin-vs-ViT bootstrap
# ============================================================

swin_aligned_df = (
    swin_patient_df
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


vit_aligned_df = (
    vit_patient_df
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


if not np.array_equal(
    swin_aligned_df[
        "patient_id"
    ].to_numpy(
        dtype=int
    ),

    vit_aligned_df[
        "patient_id"
    ].to_numpy(
        dtype=int
    ),
):
    raise RuntimeError(
        "Swin/ViT patient alignment failure."
    )


if not np.array_equal(
    swin_aligned_df[
        "y_true"
    ].to_numpy(
        dtype=int
    ),

    vit_aligned_df[
        "y_true"
    ].to_numpy(
        dtype=int
    ),
):
    raise RuntimeError(
        "Swin/ViT patient-reference mismatch."
    )


paired_y_true = (
    swin_aligned_df[
        "y_true"
    ]
    .to_numpy(
        dtype=int
    )
)


paired_swin_pred = (
    swin_aligned_df[
        "y_pred"
    ]
    .to_numpy(
        dtype=int
    )
)


paired_vit_pred = (
    vit_aligned_df[
        "y_pred"
    ]
    .to_numpy(
        dtype=int
    )
)


paired_class_indices = {
    class_index: np.flatnonzero(
        paired_y_true
        ==
        class_index
    )

    for class_index in range(
        NUM_CLASSES
    )
}


paired_rng = np.random.default_rng(
    BOOTSTRAP_SEED
    +
    50000
)


paired_bootstrap_rows = []


for bootstrap_index in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_indices = np.concatenate([
        paired_rng.choice(
            paired_class_indices[
                class_index
            ],

            size=len(
                paired_class_indices[
                    class_index
                ]
            ),

            replace=True,
        )

        for class_index in range(
            NUM_CLASSES
        )
    ])

    sampled_true = paired_y_true[
        sampled_indices
    ]

    swin_metrics = compute_multiclass_metrics(
        sampled_true,

        paired_swin_pred[
            sampled_indices
        ],
    )

    vit_metrics = compute_multiclass_metrics(
        sampled_true,

        paired_vit_pred[
            sampled_indices
        ],
    )

    paired_bootstrap_rows.append({
        "bootstrap_index": (
            int(
                bootstrap_index
            )
        ),

        "delta_qwk": (
            swin_metrics[
                "qwk"
            ]
            -
            vit_metrics[
                "qwk"
            ]
        ),

        "delta_mae": (
            swin_metrics[
                "mae"
            ]
            -
            vit_metrics[
                "mae"
            ]
        ),

        "delta_macro_f1": (
            swin_metrics[
                "macro_f1"
            ]
            -
            vit_metrics[
                "macro_f1"
            ]
        ),

        "delta_accuracy": (
            swin_metrics[
                "accuracy"
            ]
            -
            vit_metrics[
                "accuracy"
            ]
        ),
    })


paired_bootstrap_df = pd.DataFrame(
    paired_bootstrap_rows
)


atomic_csv_save(
    paired_bootstrap_df,

    OUTPUT_ROOT
    / "official_test_Swin_vs_ViT_paired_bootstrap_samples.csv",

    index=False,
)


swin_point = compute_multiclass_metrics(
    paired_y_true,
    paired_swin_pred,
)


vit_point = compute_multiclass_metrics(
    paired_y_true,
    paired_vit_pred,
)


paired_metric_columns = {
    "qwk": (
        "delta_qwk"
    ),

    "mae": (
        "delta_mae"
    ),

    "macro_f1": (
        "delta_macro_f1"
    ),

    "accuracy": (
        "delta_accuracy"
    ),
}


paired_summary_rows = []


for metric_name, column_name in paired_metric_columns.items():
    values = (
        paired_bootstrap_df[
            column_name
        ]
        .dropna()
        .to_numpy()
    )

    probability_nonpositive = float(
        np.mean(
            values <= 0
        )
    )

    probability_nonnegative = float(
        np.mean(
            values >= 0
        )
    )

    tail_proportion = min(
        1.0,

        2.0
        *
        min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    paired_summary_rows.append({
        "metric": (
            metric_name
        ),

        "difference_definition": (
            "Swin - ViT"
        ),

        "swin_point_estimate": float(
            swin_point[
                metric_name
            ]
        ),

        "vit_point_estimate": float(
            vit_point[
                metric_name
            ]
        ),

        "point_difference": float(
            swin_point[
                metric_name
            ]
            -
            vit_point[
                metric_name
            ]
        ),

        "difference_ci_low": float(
            np.percentile(
                values,
                2.5,
            )
        ),

        "difference_ci_high": float(
            np.percentile(
                values,
                97.5,
            )
        ),

        "bootstrap_two_sided_tail_proportion": (
            tail_proportion
        ),

        "direction_note": (
            "Negative favors Swin"
            if metric_name == "mae"
            else
            "Positive favors Swin"
        ),
    })


paired_summary_df = pd.DataFrame(
    paired_summary_rows
)


atomic_csv_save(
    paired_summary_df,

    OUTPUT_ROOT
    / "official_test_Swin_vs_ViT_paired_bootstrap_summary.csv",

    index=False,
)


# ============================================================
# 34. Patient per-class metrics and bootstrap CIs
# ============================================================

patient_class_point_parts = []


for model_name in [
    "Swin",
    "ViT",
]:
    patient_df = model_patient_frames[
        model_name
    ]

    patient_class_point_parts.append(
        per_class_metrics_df(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],

            model_name=(
                model_name
            ),

            evaluation_level=(
                "patient"
            ),
        )
    )


patient_class_point_df = pd.concat(
    patient_class_point_parts,
    ignore_index=True,
)


patient_class_ci_rows = []


for model_name in [
    "Swin",
    "ViT",
]:
    bootstrap_df = model_bootstrap_lookup[
        model_name
    ]

    point_df = (
        patient_class_point_df[
            patient_class_point_df[
                "model"
            ]
            ==
            model_name
        ]
        .set_index(
            "mayo_class"
        )
    )

    for class_index in range(
        NUM_CLASSES
    ):
        for metric_name in [
            "precision",
            "recall",
            "f1",
        ]:
            bootstrap_column = (
                f"class_{class_index}_{metric_name}"
            )

            values = (
                bootstrap_df[
                    bootstrap_column
                ]
                .dropna()
                .to_numpy()
            )

            patient_class_ci_rows.append({
                "model": (
                    model_name
                ),

                "mayo_class": (
                    int(
                        class_index
                    )
                ),

                "metric": (
                    metric_name
                ),

                "point_estimate": float(
                    point_df.loc[
                        class_index,
                        metric_name,
                    ]
                ),

                "ci_low": float(
                    np.percentile(
                        values,
                        2.5,
                    )
                ),

                "ci_high": float(
                    np.percentile(
                        values,
                        97.5,
                    )
                ),

                "support": int(
                    point_df.loc[
                        class_index,
                        "support",
                    ]
                ),
            })


patient_class_ci_df = pd.DataFrame(
    patient_class_ci_rows
)


atomic_csv_save(
    patient_class_point_df,

    OUTPUT_ROOT
    / "official_test_patient_per_class_metrics.csv",

    index=False,
)


atomic_csv_save(
    patient_class_ci_df,

    OUTPUT_ROOT
    / "official_test_patient_per_class_metrics_bootstrap_ci.csv",

    index=False,
)


# ============================================================
# 35. Image-level per-class metrics
# ============================================================

image_class_parts = []


for model_name in [
    "Swin",
    "ViT",
]:
    image_df = model_image_frames[
        model_name
    ]

    image_class_parts.append(
        per_class_metrics_df(
            image_df[
                "y_true"
            ],

            image_df[
                "y_pred"
            ],

            model_name=(
                model_name
            ),

            evaluation_level=(
                "image"
            ),
        )
    )


image_class_df = pd.concat(
    image_class_parts,
    ignore_index=True,
)


atomic_csv_save(
    image_class_df,

    OUTPUT_ROOT
    / "official_test_image_per_class_metrics.csv",

    index=False,
)


# ============================================================
# 36. Confusion matrices
# ============================================================

for model_name in [
    "Swin",
    "ViT",
]:
    image_df = model_image_frames[
        model_name
    ]

    patient_df = model_patient_frames[
        model_name
    ]

    atomic_csv_save(
        raw_confusion_df(
            image_df[
                "y_true"
            ],

            image_df[
                "y_pred"
            ],
        ),

        OUTPUT_ROOT
        /
        f"{model_name}_official_test_image_confusion_matrix_raw.csv",

        index=True,
    )

    atomic_csv_save(
        normalized_confusion_df(
            image_df[
                "y_true"
            ],

            image_df[
                "y_pred"
            ],
        ),

        OUTPUT_ROOT
        /
        f"{model_name}_official_test_image_confusion_matrix_row_normalized.csv",

        index=True,
    )

    atomic_csv_save(
        raw_confusion_df(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],
        ),

        OUTPUT_ROOT
        /
        f"{model_name}_official_test_patient_confusion_matrix_raw.csv",

        index=True,
    )

    atomic_csv_save(
        normalized_confusion_df(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],
        ),

        OUTPUT_ROOT
        /
        f"{model_name}_official_test_patient_confusion_matrix_row_normalized.csv",

        index=True,
    )


# ============================================================
# 37. Severe Mayo 0/1 vs 2/3 sensitivity with CI
# ============================================================

severe_binary_rows = []


for model_name in [
    "Swin",
    "ViT",
]:
    patient_df = model_patient_frames[
        model_name
    ]

    point_metrics = severe_binary_metrics(
        patient_df[
            "y_true"
        ],

        patient_df[
            "y_pred"
        ],
    )

    bootstrap_df = model_bootstrap_lookup[
        model_name
    ]

    for metric_name, point_value in point_metrics.items():
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        severe_binary_rows.append({
            "model": (
                model_name
            ),

            "metric": (
                metric_name
            ),

            "point_estimate": float(
                point_value
            ),

            "ci_low": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "ci_high": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "analysis_role": (
                "predeclared_exploratory_sensitivity"
            ),
        })


severe_binary_df = pd.DataFrame(
    severe_binary_rows
)


atomic_csv_save(
    severe_binary_df,

    OUTPUT_ROOT
    / "official_test_mayo01_vs_mayo23_sensitivity.csv",

    index=False,
)


# ============================================================
# 38. Alternative patient prediction aggregation rules
# ============================================================

CLASS_VALUES = np.arange(
    NUM_CLASSES,
    dtype=float,
)


AGGREGATION_RULES = [
    "mean_probability_argmax",
    "median_probability_argmax",
    "majority_vote_low_tie",
    "mean_expected_mes_round",
    "median_expected_mes_round",
    "maximum_predicted_class",
]


def round_half_up_class(
    value,
):
    rounded_value = np.floor(
        float(value)
        +
        0.5
    )

    return int(
        np.clip(
            rounded_value,
            0,
            NUM_CLASSES - 1,
        )
    )


def majority_vote_low_tie(
    predictions,
):
    counts = np.bincount(
        np.asarray(
            predictions,
            dtype=int,
        ),

        minlength=NUM_CLASSES,
    )

    tied_classes = np.flatnonzero(
        counts
        ==
        counts.max()
    )

    return int(
        tied_classes.min()
    )


def aggregate_all_rules(
    image_df,
):
    working_df = image_df.copy()

    probabilities = (
        working_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    working_df[
        "hard_prediction"
    ] = (
        probabilities
        .argmax(
            axis=1
        )
        .astype(int)
    )

    working_df[
        "expected_mes"
    ] = (
        probabilities
        @
        CLASS_VALUES
    )

    reference_lookup = (
        patient_reference_df
        .set_index(
            "patient_id"
        )
    )

    rows = []

    for patient_id, patient_images in working_df.groupby(
        "patient_id",
        sort=True,
    ):
        patient_probabilities = (
            patient_images[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )

        hard_predictions = (
            patient_images[
                "hard_prediction"
            ]
            .to_numpy(
                dtype=int
            )
        )

        expected_mes = (
            patient_images[
                "expected_mes"
            ]
            .to_numpy(
                dtype=float
            )
        )

        mean_probabilities = (
            patient_probabilities
            .mean(
                axis=0
            )
        )

        median_probabilities = np.median(
            patient_probabilities,
            axis=0,
        )

        predictions_by_rule = {
            "mean_probability_argmax": int(
                mean_probabilities.argmax()
            ),

            "median_probability_argmax": int(
                median_probabilities.argmax()
            ),

            "majority_vote_low_tie": (
                majority_vote_low_tie(
                    hard_predictions
                )
            ),

            "mean_expected_mes_round": (
                round_half_up_class(
                    expected_mes.mean()
                )
            ),

            "median_expected_mes_round": (
                round_half_up_class(
                    np.median(
                        expected_mes
                    )
                )
            ),

            "maximum_predicted_class": int(
                hard_predictions.max()
            ),
        }

        for aggregation_rule, prediction in predictions_by_rule.items():
            output_row = {
                "patient_id": (
                    int(
                        patient_id
                    )
                ),

                "aggregation_rule": (
                    aggregation_rule
                ),

                "y_pred": (
                    int(
                        prediction
                    )
                ),
            }

            for reference_rule in REFERENCE_RULES:
                output_row[
                    reference_rule
                ] = int(
                    reference_lookup.loc[
                        patient_id,
                        reference_rule,
                    ]
                )

            rows.append(
                output_row
            )

    return pd.DataFrame(
        rows
    )


# ============================================================
# 39. Aggregation x reference sensitivity grid
# ============================================================

sensitivity_grid_rows = []


for model_name in [
    "Swin",
    "ViT",
]:
    all_aggregation_df = aggregate_all_rules(
        model_image_frames[
            model_name
        ]
    )

    for aggregation_rule in AGGREGATION_RULES:
        aggregation_df = (
            all_aggregation_df[
                all_aggregation_df[
                    "aggregation_rule"
                ]
                ==
                aggregation_rule
            ]
        )

        for reference_rule in REFERENCE_RULES:
            metrics = compute_multiclass_metrics(
                aggregation_df[
                    reference_rule
                ],

                aggregation_df[
                    "y_pred"
                ],
            )

            sensitivity_grid_rows.append({
                "model": (
                    model_name
                ),

                "aggregation_rule": (
                    aggregation_rule
                ),

                "reference_rule": (
                    reference_rule
                ),

                "analysis_role": (
                    "primary"
                    if
                    (
                        model_name == "Swin"
                        and
                        aggregation_rule
                        ==
                        "mean_probability_argmax"
                        and
                        reference_rule
                        ==
                        "modal_low"
                    )
                    else
                    "descriptive_sensitivity_only"
                ),

                **metrics,
            })


sensitivity_grid_df = pd.DataFrame(
    sensitivity_grid_rows
)


atomic_csv_save(
    sensitivity_grid_df,

    OUTPUT_ROOT
    / "official_test_aggregation_reference_sensitivity_grid.csv",

    index=False,
)


swin_aggregation_sensitivity_df = (
    sensitivity_grid_df[
        (
            sensitivity_grid_df[
                "model"
            ]
            ==
            "Swin"
        )
        &
        (
            sensitivity_grid_df[
                "reference_rule"
            ]
            ==
            "modal_low"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


swin_reference_sensitivity_df = (
    sensitivity_grid_df[
        (
            sensitivity_grid_df[
                "model"
            ]
            ==
            "Swin"
        )
        &
        (
            sensitivity_grid_df[
                "aggregation_rule"
            ]
            ==
            "mean_probability_argmax"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


atomic_csv_save(
    swin_aggregation_sensitivity_df,

    OUTPUT_ROOT
    / "Swin_official_test_aggregation_sensitivity_modal_low.csv",

    index=False,
)


atomic_csv_save(
    swin_reference_sensitivity_df,

    OUTPUT_ROOT
    / "Swin_official_test_reference_sensitivity_mean_probability.csv",

    index=False,
)


# ============================================================
# 40. Report-ready primary results with 95% CIs
# ============================================================

report_rows = []


for _, point_row in main_point_df.iterrows():
    model_name = point_row[
        "model"
    ]

    model_ci_df = (
        marginal_ci_df[
            marginal_ci_df[
                "model"
            ]
            ==
            model_name
        ]
        .set_index(
            "metric"
        )
    )

    report_rows.append({
        "model": (
            model_name
        ),

        "role": (
            point_row[
                "role"
            ]
        ),

        "patient_qwk": float(
            point_row[
                "patient_qwk"
            ]
        ),

        "patient_qwk_ci_low": float(
            model_ci_df.loc[
                "qwk",
                "ci_low",
            ]
        ),

        "patient_qwk_ci_high": float(
            model_ci_df.loc[
                "qwk",
                "ci_high",
            ]
        ),

        "patient_mae": float(
            point_row[
                "patient_mae"
            ]
        ),

        "patient_mae_ci_low": float(
            model_ci_df.loc[
                "mae",
                "ci_low",
            ]
        ),

        "patient_mae_ci_high": float(
            model_ci_df.loc[
                "mae",
                "ci_high",
            ]
        ),

        "patient_macro_f1": float(
            point_row[
                "patient_macro_f1"
            ]
        ),

        "patient_macro_f1_ci_low": float(
            model_ci_df.loc[
                "macro_f1",
                "ci_low",
            ]
        ),

        "patient_macro_f1_ci_high": float(
            model_ci_df.loc[
                "macro_f1",
                "ci_high",
            ]
        ),

        "patient_accuracy": float(
            point_row[
                "patient_accuracy"
            ]
        ),

        "patient_accuracy_ci_low": float(
            model_ci_df.loc[
                "accuracy",
                "ci_low",
            ]
        ),

        "patient_accuracy_ci_high": float(
            model_ci_df.loc[
                "accuracy",
                "ci_high",
            ]
        ),

        "image_qwk": float(
            point_row[
                "image_qwk"
            ]
        ),

        "image_mae": float(
            point_row[
                "image_mae"
            ]
        ),

        "image_macro_f1": float(
            point_row[
                "image_macro_f1"
            ]
        ),

        "image_accuracy": float(
            point_row[
                "image_accuracy"
            ]
        ),
    })


report_main_results_df = pd.DataFrame(
    report_rows
)


MAIN_RESULTS_PATH = (
    OUTPUT_ROOT
    / "TABLE_official_test_primary_results.csv"
)


atomic_csv_save(
    report_main_results_df,
    MAIN_RESULTS_PATH,
    index=False,
)


# ============================================================
# 41. Patient-reference class distributions
# ============================================================

patient_class_distribution_rows = []


for reference_rule in REFERENCE_RULES:
    counts = (
        patient_reference_df[
            reference_rule
        ]
        .value_counts()
        .reindex(
            range(
                NUM_CLASSES
            ),
            fill_value=0,
        )
        .sort_index()
    )

    for class_index in range(
        NUM_CLASSES
    ):
        patient_class_distribution_rows.append({
            "reference_rule": (
                reference_rule
            ),

            "mayo_class": (
                int(
                    class_index
                )
            ),

            "n_patients": int(
                counts.loc[
                    class_index
                ]
            ),

            "percent_patients": float(
                100.0
                *
                counts.loc[
                    class_index
                ]
                /
                EXPECTED_TEST_PATIENT_COUNT
            ),
        })


patient_class_distribution_df = pd.DataFrame(
    patient_class_distribution_rows
)


atomic_csv_save(
    patient_class_distribution_df,

    OUTPUT_ROOT
    / "official_test_patient_reference_class_distributions.csv",

    index=False,
)


# ============================================================
# 42. Completion record
# ============================================================

completion_record = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_status": (
        "LOCKED_OFFICIAL_TEST_EVALUATION_COMPLETED"
    ),

    "pretest_lock_fingerprint": (
        EXPECTED_PRETEST_LOCK_FINGERPRINT
    ),

    "all_30_checkpoint_hashes_verified_before_test_access": (
        True
    ),

    "official_test_root": str(
        OFFICIAL_TEST_ROOT
    ),

    "official_test_manifest_sha256": (
        TEST_MANIFEST_SHA256
    ),

    "official_test_n_images": int(
        len(
            test_manifest_df
        )
    ),

    "official_test_n_patients": int(
        observed_test_patient_count
    ),

    "primary_model": (
        "Swin"
    ),

    "primary_model_name": (
        PRIMARY_SYSTEM[
            "model_name"
        ]
    ),

    "primary_configuration": (
        PRIMARY_SYSTEM[
            "representative_configuration"
        ]
    ),

    "primary_n_checkpoints": (
        15
    ),

    "comparator_model": (
        "ViT"
    ),

    "comparator_model_name": (
        COMPARATOR_SYSTEM[
            "model_name"
        ]
    ),

    "comparator_configuration": (
        COMPARATOR_SYSTEM[
            "representative_configuration"
        ]
    ),

    "comparator_n_checkpoints": (
        15
    ),

    "patient_reference": (
        "modal_low"
    ),

    "patient_aggregation": (
        "mean_probability_argmax"
    ),

    "tta": (
        "original + horizontal flip"
    ),

    "checkpoint_ensemble": (
        "equal probability mean across 15 checkpoints"
    ),

    "bootstrap_repetitions": (
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),

    "no_model_reselection_after_test_access": (
        True
    ),

    "no_checkpoint_reselection_after_test_access": (
        True
    ),

    "no_aggregation_reselection_after_test_access": (
        True
    ),

    "no_threshold_tuning_after_test_access": (
        True
    ),

    "no_calibration_after_test_access": (
        True
    ),

    "primary_results": (
        report_main_results_df
        .to_dict(
            orient="records"
        )
    ),

    "paired_swin_vs_vit": (
        paired_summary_df
        .to_dict(
            orient="records"
        )
    ),
}


COMPLETION_RECORD_PATH = (
    OUTPUT_ROOT
    / "OFFICIAL_TEST_EVALUATION_COMPLETED.json"
)


atomic_json_save(
    completion_record,
    COMPLETION_RECORD_PATH,
)


# ============================================================
# 43. Output integrity manifest
# ============================================================

final_output_paths = [
    TEST_MANIFEST_PATH,
    PATIENT_REFERENCE_PATH,
    MAIN_RESULTS_PATH,

    OUTPUT_ROOT
    / "official_test_patient_metric_bootstrap_ci.csv",

    OUTPUT_ROOT
    / "official_test_Swin_vs_ViT_paired_bootstrap_summary.csv",

    OUTPUT_ROOT
    / "official_test_patient_per_class_metrics.csv",

    OUTPUT_ROOT
    / "official_test_patient_per_class_metrics_bootstrap_ci.csv",

    OUTPUT_ROOT
    / "official_test_image_per_class_metrics.csv",

    OUTPUT_ROOT
    / "official_test_mayo01_vs_mayo23_sensitivity.csv",

    OUTPUT_ROOT
    / "official_test_aggregation_reference_sensitivity_grid.csv",

    SWIN_IMAGE_PREDICTION_PATH,
    SWIN_PATIENT_PREDICTION_PATH,

    VIT_IMAGE_PREDICTION_PATH,
    VIT_PATIENT_PREDICTION_PATH,

    COMPLETION_RECORD_PATH,
]


integrity_rows = []


for output_path in final_output_paths:
    if not output_path.exists():
        raise FileNotFoundError(
            f"Expected output missing:\n{output_path}"
        )

    integrity_rows.append({
        "filename": (
            output_path.name
        ),

        "path": str(
            output_path
        ),

        "sha256": file_sha256(
            output_path
        ),

        "size_bytes": int(
            output_path.stat().st_size
        ),
    })


output_integrity_df = pd.DataFrame(
    integrity_rows
)


OUTPUT_INTEGRITY_PATH = (
    OUTPUT_ROOT
    / "official_test_output_integrity_manifest.csv"
)


atomic_csv_save(
    output_integrity_df,
    OUTPUT_INTEGRITY_PATH,
    index=False,
)


# ============================================================
# 44. Environment metadata
# ============================================================

environment_metadata = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": (
        platform.python_version()
    ),

    "platform": (
        platform.platform()
    ),

    "torch": (
        torch.__version__
    ),

    "torchvision": (
        torchvision.__version__
    ),

    "timm": (
        timm.__version__
    ),

    "numpy": (
        np.__version__
    ),

    "pandas": (
        pd.__version__
    ),

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "pillow": (
        safe_package_version(
            "Pillow"
        )
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": (
        torch.version.cuda
    ),

    "gpu_name": (
        torch.cuda.get_device_name(
            0
        )
        if torch.cuda.is_available()
        else
        None
    ),

    "pretest_lock_fingerprint": (
        EXPECTED_PRETEST_LOCK_FINGERPRINT
    ),
}


atomic_json_save(
    environment_metadata,

    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 45. DISPLAY — Official test audit
# ============================================================

print("\n" + "=" * 130)
print("OFFICIAL TEST DATASET AUDIT")
print("=" * 130)


print(
    "Official test images:",
    len(
        test_manifest_df
    ),
)


print(
    "Official test patients:",
    observed_test_patient_count,
)


print(
    "\nImage-level class counts:"
)


display(
    test_manifest_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename(
        "n_images"
    )
    .to_frame()
)


print(
    "\nPrimary modal_low patient counts:"
)


display(
    primary_reference_counts
    .rename(
        "n_patients"
    )
    .to_frame()
)


print(
    "\nReference disagreement:"
)


display(
    reference_disagreement_df
)


# ============================================================
# 46. DISPLAY — Locked primary results
# ============================================================

print("\n" + "=" * 130)
print("LOCKED OFFICIAL-TEST PRIMARY RESULTS")
print("=" * 130)


display(
    report_main_results_df
)


# ============================================================
# 47. DISPLAY — Paired Swin vs ViT bootstrap
# ============================================================

print("\n" + "=" * 130)
print("PAIRED PATIENT BOOTSTRAP — SWIN VS ViT")
print("=" * 130)


display(
    paired_summary_df
)


# ============================================================
# 48. DISPLAY — Per-class patient metrics
# ============================================================

print("\n" + "=" * 130)

print(
    "PATIENT-LEVEL PER-CLASS METRICS "
    "WITH 95% BOOTSTRAP CIs"
)

print("=" * 130)


display(
    patient_class_ci_df
)


# ============================================================
# 49. DISPLAY — Severe-disease sensitivity
# ============================================================

print("\n" + "=" * 130)

print(
    "PREDECLARED MAYO 0/1 VS MAYO 2/3 SENSITIVITY"
)

print("=" * 130)


display(
    severe_binary_df
)


# ============================================================
# 50. DISPLAY — Aggregation sensitivity
# ============================================================

print("\n" + "=" * 130)

print(
    "SWIN AGGREGATION SENSITIVITY AGAINST MODAL_LOW"
)

print("=" * 130)


display(
    swin_aggregation_sensitivity_df
)


# ============================================================
# 51. DISPLAY — Reference sensitivity
# ============================================================

print("\n" + "=" * 130)

print(
    "SWIN REFERENCE SENSITIVITY USING "
    "LOCKED MEAN-PROBABILITY AGGREGATION"
)

print("=" * 130)


display(
    swin_reference_sensitivity_df
)


# ============================================================
# 52. Completion receipt
# ============================================================

swin_main_result = (
    report_main_results_df[
        report_main_results_df[
            "model"
        ]
        ==
        "Swin"
    ]
    .iloc[0]
)


vit_main_result = (
    report_main_results_df[
        report_main_results_df[
            "model"
        ]
        ==
        "ViT"
    ]
    .iloc[0]
)


completion_receipt_text = f"""
ROUTE A LOCKED OFFICIAL-TEST EVALUATION
=======================================

STATUS:
COMPLETED

PRE-TEST LOCK FINGERPRINT:
{EXPECTED_PRETEST_LOCK_FINGERPRINT}

ALL 30 CHECKPOINT HASHES VERIFIED BEFORE TEST ACCESS:
YES

OFFICIAL TEST IMAGES:
{len(test_manifest_df)}

OFFICIAL TEST PATIENTS:
{observed_test_patient_count}

PRIMARY MODEL:
Swin

PRIMARY PATIENT QWK:
{swin_main_result['patient_qwk']:.6f}

PRIMARY PATIENT QWK 95% CI:
[{swin_main_result['patient_qwk_ci_low']:.6f},
 {swin_main_result['patient_qwk_ci_high']:.6f}]

PRIMARY PATIENT MAE:
{swin_main_result['patient_mae']:.6f}

PRIMARY PATIENT MAE 95% CI:
[{swin_main_result['patient_mae_ci_low']:.6f},
 {swin_main_result['patient_mae_ci_high']:.6f}]

PRIMARY PATIENT MACRO F1:
{swin_main_result['patient_macro_f1']:.6f}

PRIMARY PATIENT ACCURACY:
{swin_main_result['patient_accuracy']:.6f}

PRIMARY IMAGE QWK:
{swin_main_result['image_qwk']:.6f}

PRIMARY IMAGE MAE:
{swin_main_result['image_mae']:.6f}

PRESPECIFIED COMPARATOR:
ViT

ViT PATIENT QWK:
{vit_main_result['patient_qwk']:.6f}

ViT PATIENT MAE:
{vit_main_result['patient_mae']:.6f}

PATIENT REFERENCE:
modal_low

PATIENT AGGREGATION:
mean_probability_argmax

PRIMARY INFERENCE:
15-checkpoint equal probability ensemble

TTA:
original + horizontal flip

BOOTSTRAP:
5,000 stratified patient replicates

MODEL RESELECTION AFTER TEST ACCESS:
NO

CHECKPOINT RESELECTION AFTER TEST ACCESS:
NO

AGGREGATION RESELECTION AFTER TEST ACCESS:
NO

THRESHOLD TUNING:
NO

CALIBRATION:
NO

IMPORTANT:
These are the locked Route A official-test results.
No test-result-driven model, checkpoint, aggregation,
reference, preprocessing, TTA, weighting, threshold,
or calibration changes are permitted.
"""


COMPLETION_RECEIPT_PATH = (
    OUTPUT_ROOT
    / "LOCKED_OFFICIAL_TEST_EVALUATION_RECEIPT.txt"
)


atomic_text_save(
    completion_receipt_text.strip()
    +
    "\n",

    COMPLETION_RECEIPT_PATH,
)


# ============================================================
# 53. FINAL STATUS
# ============================================================

print("\n" + "=" * 130)

print(
    "09_locked_official_test_evaluation "
    "COMPLETED SUCCESSFULLY"
)

print("=" * 130)


print(
    "\nPRE-TEST LOCK FINGERPRINT:"
)

print(
    EXPECTED_PRETEST_LOCK_FINGERPRINT
)


print(
    "\nAll 30 checkpoint hashes verified before test access:",
    True,
)


print(
    "\nOfficial test images:",
    len(
        test_manifest_df
    ),
)


print(
    "Official test patients:",
    observed_test_patient_count,
)


print(
    "\nPrimary model:",
    "Swin",
)


print(
    "Primary patient reference:",
    "modal_low",
)


print(
    "Primary aggregation:",
    "mean_probability_argmax",
)


print(
    "Primary inference:",
    "15-checkpoint equal-probability ensemble",
)


print(
    "\nMain results:"
)

print(
    MAIN_RESULTS_PATH
)


print(
    "\nPaired Swin-vs-ViT results:"
)

print(
    OUTPUT_ROOT
    / "official_test_Swin_vs_ViT_paired_bootstrap_summary.csv"
)


print(
    "\nPer-class patient results:"
)

print(
    OUTPUT_ROOT
    / "official_test_patient_per_class_metrics_bootstrap_ci.csv"
)


print(
    "\nSevere Mayo 0/1 vs 2/3 sensitivity:"
)

print(
    OUTPUT_ROOT
    / "official_test_mayo01_vs_mayo23_sensitivity.csv"
)


print(
    "\nAggregation/reference sensitivity:"
)

print(
    OUTPUT_ROOT
    / "official_test_aggregation_reference_sensitivity_grid.csv"
)


print(
    "\nCompletion record:"
)

print(
    COMPLETION_RECORD_PATH
)


print(
    "\nCompletion receipt:"
)

print(
    COMPLETION_RECEIPT_PATH
)


print(
    "\nIMPORTANT:"
)

print(
    "The official test set has now been evaluated."
)

print(
    "Do NOT tune, reselect, recalibrate, reweight, "
    "or otherwise modify Route A based on these results."
)